# Week 1 — Activation Extraction

**RQ1: Temporal Horizon Detection — Experimental Setup**

Extracts last-token residual stream activations at every layer from `meta-llama/Meta-Llama-3.1-8B`
for all 300 prompts in the verified implicit dataset (no surface temporal cues).

Outputs `activations_immediate.pt` and `activations_long_term.pt` of shape `[300, n_layers, d_model]`
— raw material for Week 2 linear probe training and CAA steering vectors.

In [ ]:
%%bash
git clone https://github.com/Avi161/temporal-awareness.git
cd temporal-awareness
git checkout research/rq1-week1-extraction
pip install -e '.[dev]' -q

In [ ]:
import os
import sys

os.chdir('/content/temporal-awareness')
sys.path.insert(0, '/content/temporal-awareness')

import json
from pathlib import Path

import torch
from tqdm import tqdm

from src.inference import ModelRunner
from src.inference.backends import ModelBackend
from src.intertemporal.common.project_paths import get_experiment_dir
from scripts.week1.activation_extractor import (
    load_implicit_dataset,
    build_prompts,
    extract_activations_for_prompts,
    save_extraction_results,
)

In [ ]:
# Primary model — 16 GB VRAM recommended
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"
# MODEL_NAME = "mistralai/Mistral-7B-v0.1"  # fallback if VRAM constrained

DATASET_PATH = Path("data/raw/temporal_scope/temporal_scope_implicit_backup_300.json")
OUT_DIR = get_experiment_dir() / "week1_activation_extraction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
pairs, metadata = load_implicit_dataset(DATASET_PATH)
print("Dataset metadata:")
print(json.dumps(metadata, indent=2))
print(f"\nLoaded {len(pairs)} pairs")
print("\nFirst 3 pairs:")
for p in pairs[:3]:
    print(p)

In [ ]:
immediate_prompts, long_term_prompts, categories = build_prompts(pairs)
print("--- Immediate prompt [0] ---")
print(immediate_prompts[0])
print("\n--- Long-term prompt [0] ---")
print(long_term_prompts[0])
print(f"\nCategories (first 5): {categories[:5]}")

In [ ]:
runner = ModelRunner(
    MODEL_NAME,
    backend=ModelBackend.TRANSFORMERLENS,
    dtype=torch.float16,
)
print(f"n_layers = {runner.n_layers}")
print(f"d_model  = {runner.d_model}")

In [ ]:
all_names = runner.get_all_names_for_internals()
resid_names = [n for n in all_names if "hook_resid_post" in n]
print(f"Found {len(resid_names)} hook_resid_post layers")
print("First 3:", resid_names[:3])
print("Last 3: ", resid_names[-3:])

In [ ]:
test_prompt = immediate_prompts[0]
logits, cache = runner.run_with_cache(
    test_prompt,
    names_filter=lambda n: "hook_resid_post" in n,
)
sample_act = cache["blocks.0.hook_resid_post"][0, -1, :]
print(f"Logits shape:           {logits.shape}")
print(f"Sample activation shape: {sample_act.shape}  (d_model={runner.d_model})")
assert sample_act.shape[0] == runner.d_model, "d_model mismatch"
assert torch.isfinite(sample_act).all(), "Non-finite values in sample activation"
print("Sanity check passed.")

In [ ]:
acts_immediate = extract_activations_for_prompts(runner, immediate_prompts, desc="Immediate")
print(f"Shape: {acts_immediate.shape}")  # expected [300, n_layers, d_model]

In [ ]:
acts_long_term = extract_activations_for_prompts(runner, long_term_prompts, desc="Long-term")
print(f"Shape: {acts_long_term.shape}")  # expected [300, n_layers, d_model]

In [ ]:
save_extraction_results(
    OUT_DIR,
    acts_immediate,
    acts_long_term,
    categories,
    pairs,
    MODEL_NAME,
    DATASET_PATH,
)
print(f"Saved to: {OUT_DIR}")
for f in sorted(OUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt

# Verification: per-layer L2 norm distribution — should be smooth
norms_imm = acts_immediate.norm(dim=-1).mean(dim=0).cpu()  # [n_layers]
norms_lt  = acts_long_term.norm(dim=-1).mean(dim=0).cpu()

plt.figure(figsize=(10, 4))
plt.plot(norms_imm, label="immediate")
plt.plot(norms_lt,  label="long-term")
plt.xlabel("Layer")
plt.ylabel("Mean L2 norm")
plt.legend()
plt.title("Activation norm per layer")
plt.tight_layout()
plt.show()

# Verify finite values
assert torch.isfinite(acts_immediate).all(), "Non-finite values in acts_immediate"
assert torch.isfinite(acts_long_term).all(), "Non-finite values in acts_long_term"
print("All values finite: OK")

In [ ]:
# Steering vector preview: mean activation difference per layer
# Norm should peak in middle-to-late layers (consistent with interpretability literature)
diff = (acts_long_term - acts_immediate).mean(dim=0)  # [n_layers, d_model]
diff_norms = diff.norm(dim=-1).cpu()

plt.figure(figsize=(10, 4))
plt.plot(diff_norms)
plt.xlabel("Layer")
plt.ylabel("||mean(long-term) - mean(immediate)||")
plt.title("CAA steering vector magnitude per layer")
plt.tight_layout()
plt.show()

peak_layer = diff_norms.argmax().item()
print(f"Peak CAA magnitude at layer {peak_layer} (of {runner.n_layers})")